# 16. metric-aware calibration 랩 (M6)

> 선행 노트북: [15_gbm_lab.ipynb](15_gbm_lab.ipynb)
> 선행 설계서: [04-final-solution-blueprint.md](../docs/design/04-final-solution-blueprint.md) 7.1·7.2절,
> [06-preprocessing-guardrails.md](../docs/design/06-preprocessing-guardrails.md) 2.2·5.3·5.4절

설계서 04 7.1절의 **M6 — metric-aware calibration**을 실행한다. 작업 모델은 노트북 15가
채택한 `lgbm_pooled`다.

**연구 질문**

> 1. 모델을 GBM으로 바꾼 뒤에도 개선분이 FICR에서 나오는가?
> 2. 정산 구간 경계를 겨냥한 보정에 남은 여지가 얼마나 있는가?
> 3. 그 여지를 표본 밖에서 실제로 가져올 수 있는가?

**첫 질문이 이 노트북의 전제 점검이다.** M6를 다음 작업으로 고른 근거는 노트북 11의
"개선분의 79.6%가 FICR에서 나왔다"와 노트북 13의 "개선분은 어느 fold에서도 FICR이
주도한다"인데, **둘 다 RandomForest 시절의 관찰이다.** 노트북 15는 GBM으로 갈면서
`one_minus_nmae`와 `ficr`를 계산해 두고도 표시하지 않았다(노트북 15 7-3절 발견 3).
전제가 무너졌다면 M6의 설계 근거부터 다시 써야 하므로 3절에서 이것을 먼저 본다.

**둘째 질문에 오라클 상한이 필요한 이유.** 보정은 파라미터가 1~5개뿐이라 Δ가 작게
나올 것이 거의 확실하다. 그때 "보정이 무력한 것"과 "우리 후보가 약한 것"을 구분할
장치가 없으면 어느 쪽으로도 결론을 낼 수 없다. 5절이 **행별 이상 보정의 천장**을 재고,
9절이 그 천장과 실제 후보 사이의 간격을 세 층으로 쪼갠다.

**셋째 질문이 판정이다.** fit은 학습 창 내부 OOF에서만 하고(Decision Box ㉗), 채점은
F0~F8 아홉 fold에서 설계서 06 5.4절 규칙으로 한다. 채택 조건은 결과를 보기 전에
Decision Box ㉙에 못 박아 둔다 — 사후에 규칙을 고르면 규칙이 아니다.

---

**실행 방법.** 이 노트북은 로컬과 Colab 양쪽에서 같은 파일로 돈다.

| 환경 | 준비 |
|------|------|
| 로컬 | 저장소 안에서 열면 된다. 부트스트랩은 버전만 보고하고 넘어간다 |
| Colab | Drive `baram-data/open/`에 원자료 4개를 두고 첫 셀부터 순서대로 실행한다 |

학습이 30회(RF 6 + pooled 6 + inner OOF 18)이고 전부 CPU 바운드다. **Colab 무료 런타임은
2코어**라 22코어 로컬보다 몇 배 느리므로 로컬 실행을 권한다. 반복 실행에는
`scripts/run_notebook_live.py`가 셀마다 경과 시간을 찍고 매 셀 저장한다.

In [ ]:
# 0) 실행 환경 부트스트랩 — Colab에서만 동작하고 로컬에서는 아무것도 하지 않는다
import importlib
import importlib.metadata as importlibMetadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
  import google.colab  # noqa: F401
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

# 수치에 영향을 주는 넷만 고정한다. 3절의 F0 재현 게이트(1e-9)가 여기에 걸린다.
# CatBoost는 이 노트북이 쓰지 않으므로 고정 대상에서 뺐다.
PINS = {
  "numpy": "1.26.4",
  "pandas": "3.0.3",
  "scikit-learn": "1.9.0",
  "lightgbm": "4.7.0",
}
MODULE_OF = {
  "numpy": "numpy",
  "pandas": "pandas",
  "scikit-learn": "sklearn",
  "lightgbm": "lightgbm",
}
REPO_URL = "https://github.com/Dacon-Organization/baram-2026-wind-power-forecasting.git"
# 이 노트북이 머지되는 커밋을 미리 알 수 없으므로 None으로 둔다. 부트스트랩이 기본
# 브랜치의 SHA를 보고하고, 실행 결과를 반영하는 후속 PR에서 **그 실행이 실제로 쓴 SHA**로
# 채운다. 노트북 15는 삭제된 브랜치의 커밋을 박아 두어 Colab에서 도달하지 못했다.
PINNED_COMMIT = None
REPO_DIR = Path("/content/baram")
DRIVE_OPEN = Path("/content/drive/MyDrive/baram-data/open")
REQUIRED_FILES = ["train/train_labels.csv", "train/ldaps_train.csv", "train/gfs_train.csv", "info.xlsx"]


def distributionVersion(distributionName):
  # 설치 여부 판단용. dist-info가 중복이면 부정확하므로 설치 결정에만 쓴다.
  try:
    return importlibMetadata.version(distributionName)
  except Exception:
    return None


def loadedVersion(moduleName):
  # 실제로 import되는 버전. 결과를 좌우하는 것은 이쪽이다.
  # ImportError만 잡으면 부족하다 — numpy를 갈아끼운 뒤의 import는 ValueError로 죽는다.
  try:
    return getattr(importlib.import_module(moduleName), "__version__", None)
  except Exception:
    return None


def reportVersions(allowImport):
  """고정본과 어긋난 것을 {이름: (실제 버전, 출처)}로 돌려준다.

  allowImport=False면 아직 import되지 않은 것을 새로 부르지 않고 dist metadata만 본다.
  numpy를 갈아끼운 직후에는 확장 모듈을 하나라도 새로 import하는 순간
  `ValueError: numpy.dtype size changed`로 죽는다. Colab에서 실제로 났다 — 메모리에는
  런타임이 미리 올려둔 numpy 2.0.2가, 디스크에는 방금 깐 1.26.4가 있었다.
  """
  drifted = {}
  for distributionName, pin in PINS.items():
    inMemory = sys.modules.get(MODULE_OF[distributionName])
    if inMemory is not None:
      actual, source = getattr(inMemory, "__version__", None), "로드됨"
    elif allowImport:
      actual, source = loadedVersion(MODULE_OF[distributionName]), "로드됨"
    else:
      actual, source = distributionVersion(distributionName), "설치본"
    if actual != pin:
      drifted[distributionName] = (actual, source)
    print(f"  {'OK ' if actual == pin else 'BAD'}  {distributionName:14s} {str(actual):8s} ({source}, 기대 {pin})")
  return drifted


if not IN_COLAB:
  print("로컬 환경 — 부트스트랩을 건너뛴다")
  # 로컬은 버전을 갈아끼우지 않으므로 import해서 봐도 안전하다. dist-info가 중복인
  # 환경에서는 metadata보다 이쪽이 정확하다 (설치본 1.5.1 / 실제 1.9.0을 봤다).
  reportVersions(allowImport=True)
else:
  from google.colab import drive

  drive.mount("/content/drive")

  if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
  if PINNED_COMMIT is not None:
    pinnedCheckout = subprocess.run(
      ["git", "-C", str(REPO_DIR), "checkout", "--quiet", PINNED_COMMIT],
      capture_output=True, text=True,
    )
    reached = pinnedCheckout.returncode == 0
  else:
    reached = False
  currentHead = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True,
  ).stdout.strip()
  if reached:
    codeState = f"{currentHead} (고정 커밋)"
  elif PINNED_COMMIT is None:
    codeState = f"{currentHead} (기본 브랜치 — PINNED_COMMIT 미지정, 이 SHA를 기록할 것)"
  else:
    codeState = f"{currentHead} (고정 커밋에 닿지 못함 — 기본 브랜치, 코드가 다를 수 있음)"

  # data/raw/open 은 실제 디렉터리다 (MANIFEST.md·README.md가 커밋 대상).
  # Path.unlink()는 IsADirectoryError를 내므로 rmtree로 지운 뒤 링크를 건다.
  openDir = REPO_DIR / "data" / "raw" / "open"
  if openDir.is_symlink():
    openDir.unlink()
  elif openDir.exists():
    shutil.rmtree(openDir)
  openDir.parent.mkdir(parents=True, exist_ok=True)
  openDir.symlink_to(DRIVE_OPEN, target_is_directory=True)

  # 설치 결정은 metadata로 한다 — 아직 import하지 않은 것을 재보려면 이 방법뿐이다.
  toInstall = [f"{n}=={p}" for n, p in PINS.items() if distributionVersion(n) != p]
  if toInstall:
    print(f"설치: {toInstall}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *toInstall], check=True)
  if distributionVersion("seaborn") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn"], check=True)

  # 한글 폰트 — 아래 fontCandidates 중 Colab에서 잡히는 것은 NanumGothic 뿐이다.
  if not list(Path("/usr/share/fonts/truetype").glob("nanum*")):
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=True)
  fontCache = Path.home() / ".cache" / "matplotlib"
  if fontCache.exists():
    shutil.rmtree(fontCache)
  if "matplotlib" in sys.modules:
    import matplotlib.font_manager as alreadyLoadedFontManager

    for nanumFont in Path("/usr/share/fonts/truetype").glob("nanum*/*.ttf"):
      alreadyLoadedFontManager.fontManager.addfont(str(nanumFont))

  os.chdir(REPO_DIR / "notebooks")

  print(f"코드      : {codeState}")
  print(f"작업 경로 : {Path.cwd()}")
  for relativePath in REQUIRED_FILES:
    path = openDir / relativePath
    size = path.stat().st_size if path.is_file() else 0
    print(f"  {'OK     ' if path.is_file() else 'MISSING'}  {relativePath:24s} {size:>13,} bytes")
  # 설치 직후다. 여기서는 import하지 않는다 — 아래 재시작 안내를 내는 것이 이 검사의
  # 목적인데, 새 import가 ABI 오류로 먼저 죽으면 안내가 나오지 않는다.
  drifted = reportVersions(allowImport=False)

  # pip로 깔아도 이미 import된 모듈은 바뀌지 않는다. Colab 런타임은 numpy를 미리 올려두므로
  # 첫 실행에서는 거의 항상 여기에 걸린다. 재시작 없이 진행하면 ABI 오류가 난다.
  if drifted:
    needsRestart = [name for name, (_, source) in drifted.items() if source == "로드됨"]
    print()
    print("=" * 68)
    for distributionName, (actual, source) in drifted.items():
      print(f"  {distributionName}: {source} {actual} != 고정본 {PINS[distributionName]}")
    if needsRestart:
      print("  런타임 > 세션 다시 시작 을 누른 뒤 이 셀을 다시 실행하세요.")
      print("  clone·Drive 마운트·설치는 남아 있어 두 번째 실행은 몇 초면 끝납니다.")
    else:
      reinstallSpec = " ".join(f"{name}=={PINS[name]}" for name in drifted)
      print("  설치본이 고정본과 다릅니다 — 재시작으로는 풀리지 않습니다. 다음을 실행하세요:")
      print(f"    !pip install -q --force-reinstall {reinstallSpec}")
    print("=" * 68)
    raise RuntimeError("환경이 고정본과 다릅니다 — 위 안내를 따른 뒤 이 셀을 다시 실행하세요")

부트스트랩은 **같은 파일이 로컬과 Colab 양쪽에서 돌게 하려고** 있다. 노트북을 환경별로
갈라 두면 어느 쪽이 정본인지 모르게 되고, 그 순간 재현 주장이 무너진다.

**노트북 15와 달라진 곳이 두 군데다.**

`PINNED_COMMIT`을 `None`으로 뒀다. 이 노트북은 코드가 완성된 시점과 실행 시점이 다르고,
자신이 머지될 커밋을 미리 알 수 없다. 노트북 15는 브랜치 커밋을 박아 뒀는데 squash
머지로 그 브랜치가 지워져 Colab에서 도달하지 못했다. 그래서 여기서는 기본 브랜치의
SHA를 **보고하게** 하고, 실행 결과를 반영하는 후속 PR에서 그 SHA를 박는다.
기록해야 하는 것은 "쓰고 싶었던 커밋"이 아니라 **실제로 쓴 커밋**이다.

CatBoost를 고정 목록에서 뺐다. 노트북 15가 `cat_pooled`를 제출 후보로만 남기고 작업
모델을 `lgbm_pooled`로 정했으므로 이 노트북은 CatBoost를 쓰지 않는다. 쓰지 않는 것을
고정하면 설치 시간만 늘고 실패 지점이 하나 늘어난다.

**게이트는 예외를 던지지 않는다.** 3절의 `reproductionOk`는 계산해 표시만 하므로,
버전이 어긋나도 노트북은 끝까지 돌고 `통과 : False`만 조용히 찍는다. 그 줄을 직접 볼 것.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import font_manager


def resolveProjectRoot():
  here = Path.cwd().resolve()
  for candidate in [here, *here.parents]:
    if (candidate / "src" / "baram").is_dir():
      return candidate
  raise RuntimeError("src/baram을 찾지 못했습니다")


def resolveOfficialDataDir(projectRoot):
  for candidate in [projectRoot, *projectRoot.parents]:
    dataDir = candidate / "data" / "raw" / "open"
    if (dataDir / "train" / "train_labels.csv").is_file():
      return dataDir
  raise RuntimeError("공식 데이터 디렉터리를 찾지 못했습니다")


projectRoot = resolveProjectRoot()
if str(projectRoot / "src") not in sys.path:
  sys.path.insert(0, str(projectRoot / "src"))
dataDir = resolveOfficialDataDir(projectRoot)

sns.set_theme(style="whitegrid")
fontCandidates = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "DejaVu Sans"]
availableFonts = {font.name for font in font_manager.fontManager.ttflist}
selectedFont = next((font for font in fontCandidates if font in availableFonts), "DejaVu Sans")
plt.rcParams["font.family"] = selectedFont
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

import lightgbm
import sklearn

print(f"project root : {projectRoot.name}")
print(f"official data: {dataDir}")
print(f"lightgbm     : {lightgbm.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"한글 폰트    : {selectedFont}")

09~15와 같은 탐색 규약이다. 이 노트북은 `outputs/`에 아무것도 쓰지 않는다.

---

## 1. 후보와 fold를 먼저 못 박는다

설계서 06 5.1절이 경고한 것은 "GBM·fold·calibration으로 후보가 수십 개가 되는" 상황이다.
calibration은 후보를 만들기가 특히 쉽다 — 함수 형태 × 목적함수 × 그리드 폭 × fit 범위가
곱으로 불어난다. 그래서 **모델과 피처셋을 고정하고 보정만 6종으로 못 박는다.**

| 보정 후보 | 형태 | 목적함수 | 역할 |
|-----------|------|----------|------|
| `none` | 항등 | — | **통제군** — 보정하지 않은 `lgbm_pooled` |
| `bias_mae` | `p̂ + b` | NMAE 최소 | metric-aware의 대조군 |
| `bias_metric` | `p̂ + b` | total_score 최대 | 가장 단순한 metric-aware |
| `affine_metric` | `a·p̂ + b` | total_score 최대 | 수준 + 기울기 |
| `binned_metric` | 예측 5분위별 `p̂ + b_k` | total_score 최대 | 조건부 편향 교정 |
| `isotonic` | isotonic `p̂ → ŷ` | 단조·제곱오차 | 유연하지만 metric-aware 아님 |

**`bias_mae`와 `bias_metric`이 이 노트북의 핵심 대조다.** 함수 형태가 같고 그리드가 같고
fit 데이터가 같다. 다른 것은 목적함수 하나뿐이다. 두 결과가 갈리지 않으면 "metric-aware"는
이름뿐이며, 그 사실을 확인하는 것 자체가 결과다.

보정은 전부 **설비용량 정규화 공간**(`p̂ = 예측 / 설비용량`)에서 정의한다. 설비용량이
다른 세 그룹이 파라미터 하나를 공유할 수 있고, Group 3 라벨이 0행인 학습 창(`W_2022`)에서도
보정이 정의된다. 모델은 `lgbm_pooled`, 피처셋은 `spatial_idw2_all_group` 하나로 고정한다.

비교 기준으로 챔피언 `rf_spatial`도 함께 학습한다. 3절의 지표 분해에 필요하고,
노트북 14·15가 기록한 F0 값을 그대로 내는지가 **재현 게이트**가 된다.

In [ ]:
from baram.calibration import (
  BIAS_GRID, BINNED_BINS, CALIBRATORS, DEFAULT_OOF_BLOCKS, SLOPE_GRID,
  apply_calibration, band_transition, boundary_density, decompose_normalized,
  describe_oof_blocks, fit_calibration, fold_normalized_terms, inner_oof_blocks,
  nmae_normalized, normalized_terms, oracle_row_bound, score_normalized,
)
from baram.feature_config import get_feature_set
from baram.features.turbine_metadata import load_turbine_locations
from baram.folds import (
  FOLDS, RANK_THRESHOLD, TRAIN_WINDOWS, describe_folds, group_error_terms,
  paired_bootstrap_gap, run_window, score_fold, scoreable_targets, total_from_terms,
  window_predict_ranges,
)
from baram.gbm import LIGHTGBM_PARAMS, make_lightgbm, run_window_pooled
from baram.metrics import CAPACITY_KWH, TARGET_COLS

readOfficial = lambda relativePath: pd.read_csv(dataDir / relativePath, encoding="utf-8-sig")
trainLabels = readOfficial("train/train_labels.csv")
trainLabels["kst_dtm"] = pd.to_datetime(trainLabels["kst_dtm"])
ldapsTrain = readOfficial("train/ldaps_train.csv")
gfsTrain = readOfficial("train/gfs_train.csv")
ldapsTrain["forecast_kst_dtm"] = pd.to_datetime(ldapsTrain["forecast_kst_dtm"])
gfsTrain["forecast_kst_dtm"] = pd.to_datetime(gfsTrain["forecast_kst_dtm"])
turbines = load_turbine_locations(dataDir / "info.xlsx")

CHAMPION, WORKING = "rf_spatial", "lgbm_pooled"
SPATIAL = "spatial_idw2_all_group"
CONTROL = "none"
spatialConfig = get_feature_set(SPATIAL)
foldOrder = list(FOLDS)
windowRanges = window_predict_ranges()
windowOrder = list(windowRanges)
foldScoreColumns = {name: scoreable_targets(trainLabels, name) for name in foldOrder}

print("fold 정의 — 노트북 13·14가 확정한 F0~F8을 그대로 쓴다")
display(describe_folds(trainLabels)[
  ["방향", "검증 연도", "train 창", "train 행", "G3 train 라벨", "valid 행", "채점 그룹"]
])

oofFrame = pd.concat(
  [describe_oof_blocks(trainLabels, name, TRAIN_WINDOWS[name], blocks=DEFAULT_OOF_BLOCKS)
   for name in windowOrder],
  ignore_index=True,
)

print(f"보정 후보   : {len(CALIBRATORS)}종 사전 등록 — {list(CALIBRATORS)}")
print(f"모델·피처셋 : {WORKING} · {SPATIAL} (고정)")
print(f"학습 창     : {len(windowOrder)}종 · 창당 OOF 블록 {DEFAULT_OOF_BLOCKS - 1}개")
print(f"학습 횟수   : outer {2 * len(windowOrder)}회 + inner OOF {(DEFAULT_OOF_BLOCKS - 1) * len(windowOrder)}회"
      f" = {3 * len(windowOrder) + len(windowOrder)}회")
print(f"bias 그리드 : {BIAS_GRID.min():+.3f} ~ {BIAS_GRID.max():+.3f}"
      f" step {BIAS_GRID[1] - BIAS_GRID[0]:.3f} ({BIAS_GRID.size}점)")
print(f"slope 그리드: {SLOPE_GRID.min():.2f} ~ {SLOPE_GRID.max():.2f}"
      f" step {SLOPE_GRID[1] - SLOPE_GRID[0]:.2f} ({SLOPE_GRID.size}점)")
print(f"구간 보정   : 예측 {BINNED_BINS}분위")
print()
print("학습 창 내부 OOF 분할 — 겹침 행이 전부 0이어야 한다")
display(oofFrame)
print(f"겹침 행 합계 : {int(oofFrame['겹침 행'].sum())}  (0이 아니면 아래 학습을 진행하지 말 것)")
print()
print("LightGBM 고정 파라미터 (노트북 15와 동일)")
display(pd.Series(LIGHTGBM_PARAMS).to_frame("값"))

### Decision Box ㉗ — 보정은 학습 창 내부 OOF로만 fit한다

**선택지**

- (A) 검증 fold의 예측·실측으로 직접 보정을 fit한다
- (B) **학습 창을 시간 순 4블록으로 나눠 expanding-window OOF를 만들고 거기서만 fit한다**
- (C) 학습 창의 마지막 1/3만 holdout으로 떼어 거기서 fit한다

**근거**

(A)는 설계서 06 2.2절의 **절대 금지**다. 평가 대상 구간의 값으로 통계를 fit하는 것이며,
보정처럼 파라미터가 적은 변환에서는 이 누수가 특히 달다 — 점수는 확실히 오르고
제출에서는 재현되지 않는다. 설계서 04 7.2절도 "fold 내부 OOF로 fit한 보정만 허용"으로
이미 못 박았다. 다만 (A)를 **상한 진단으로는** 쓴다. 9절이 "함수 형태의 천장"을
재는 데 필요하고, 누수임을 표에 명시한 채로만 쓴다.

(C)는 추가 학습이 6회로 싸지만 OOF 표본이 창의 1/3뿐이다.

(B)를 고르면서 **블록 수를 3으로 둔 이유는 "더 쪼개면 더 안정적"이 아니기 때문이다.**
6블록으로 늘리면 OOF 표본은 75% → 83%로 10%밖에 안 늘지만(파라미터 1~5개를 맞추는 데
이미 수천 행이다) inner 학습 크기가 17%까지 내려간다. **안정성을 결정하는 것은 OOF 행
수가 아니라 inner 모델과 최종 모델의 괴리**이고, 더 쪼개면 그 괴리가 커진다.

**채택: (B)** — 그리고 블록 수를 늘리는 대신 **추가 학습 0회로 민감도를 직접 보인다.**
8절에서 같은 OOF 예측을 (a) 3블록 전체와 (b) 마지막 블록만(inner 학습 75% — 최종 모델과
가장 잘 맞는 조각)으로 각각 다시 fit해 파라미터와 판정을 나란히 놓는다. 두 판정이 같으면
OOF 분할 방식이 결론을 만들지 않았다는 것이 **가정이 아니라 출력**으로 남는다.

**남는 한계를 미리 적는다.** inner 모델은 최종 모델보다 학습 데이터가 적으므로 오차
분포가 더 넓다. 그 분포에 맞춘 보정은 최종 모델에 조금 과교정이며, 이 방향의 편향은
8절의 민감도로 크기를 재되 없앨 수는 없다.

### Decision Box ㉘ — 목적함수는 FICR이 아니라 total_score다

**선택지**

- (A) FICR만 최대화한다 — 계단 함수가 지렛대라면 그것을 직접 겨냥한다
- (B) **total_score를 최대화한다**
- (C) FICR을 최대화하되 1-NMAE 악화에 상한을 두는 제약 최적화

**근거**

(A)는 설계서 04 7.2절의 후보 선택 규칙("FICR 개선 — 1-NMAE가 과도하게 악화되면 제외")을
목적함수 단계에서 이미 위반한다. FICR은 오차율 6% 안에서는 **완전히 평평하다** — 오차
1%와 5.9%가 같은 4원이다. 그래서 FICR만 보면 이미 안전한 행의 정확도를 얼마든지 버려도
손실이 없고, 그 손실은 1-NMAE에서만 나타난다. 대회 점수는 둘의 평균이다.

(C)는 (B)와 같은 것을 더 복잡하게 하는 방법이다. total_score 자체가 이미 두 항의
가중합이므로 상한을 따로 둘 이유가 없다.

**채택: (B)** — `0.5 × (1 − NMAE) + 0.5 × FICR`. 대회가 채점하는 그 식이다.
`bias_mae`를 후보로 남겨 둔 것은 이 선택의 대조군이다 — 같은 함수 형태에서 목적함수만
NMAE로 바꿨을 때 결과가 갈리는지를 7절에서 직접 본다.

**하이퍼파라미터에 해당하는 그리드도 고정한다.** bias는 `±0.06` step `0.001`,
slope는 `0.80~1.20` step `0.01`이다. fold 성적을 보고 넓히지 않는다 — 넓히는 순간
그 fold는 검증셋이 아니라 학습셋이 된다(노트북 15 Decision Box ㉔와 같은 이유).

---

## 2. 학습 — 30회

챔피언 RF 6회, 작업 모델 pooled 6회, inner OOF pooled 18회다. 같은 학습 창을 쓰는 fold는
학습을 공유한다 — 행 간 연산이 0건이므로 한 번 예측한 뒤 검증 구간별로 잘라 써도
개별 실행과 결과가 같다(`folds.py`).

In [ ]:
predictionCache, oofPredictions, oofSplits, timings = {}, {}, {}, []
totalStarted = time.perf_counter()

for windowName, (predictStart, predictEnd) in windowRanges.items():
  started = time.perf_counter()
  championRun = run_window(
    spatialConfig, windowName, predictStart, predictEnd,
    labels=trainLabels, ldaps=ldapsTrain, gfs=gfsTrain, turbine_locations=turbines,
  )
  championSeconds = time.perf_counter() - started
  predictionCache[(CHAMPION, windowName)] = championRun.predictions
  timings.append({"단계": "outer RF", "대상": CHAMPION, "창": windowName, "초": championSeconds})

  started = time.perf_counter()
  workingRun = run_window_pooled(
    spatialConfig, windowName, predictStart, predictEnd,
    labels=trainLabels, ldaps=ldapsTrain, gfs=gfsTrain, turbine_locations=turbines,
    make_estimator=make_lightgbm,
  )
  workingSeconds = time.perf_counter() - started
  predictionCache[(WORKING, windowName)] = workingRun.predictions
  timings.append({"단계": "outer pooled", "대상": WORKING, "창": windowName, "초": workingSeconds})

  trainedGroups = ",".join(target[-1] for target in workingRun.trained_targets)
  print(f"  outer  {windowName:14s} RF {championSeconds:7.1f}s · pooled {workingSeconds:7.1f}s"
        f"  pooled 학습G{trainedGroups}  피처 {workingRun.feature_count}")

print()
for windowName in windowOrder:
  splits = inner_oof_blocks(*TRAIN_WINDOWS[windowName], blocks=DEFAULT_OOF_BLOCKS)
  oofSplits[windowName] = splits
  for split in splits:
    started = time.perf_counter()
    run = run_window_pooled(
      spatialConfig, f"{windowName}#B{split.block}", split.predict_start, split.predict_end,
      labels=trainLabels, ldaps=ldapsTrain, gfs=gfsTrain, turbine_locations=turbines,
      make_estimator=make_lightgbm, train_bounds=split.bounds,
    )
    elapsed = time.perf_counter() - started
    oofPredictions[(windowName, split.block)] = run.predictions
    timings.append({"단계": "inner OOF", "대상": f"{windowName}#B{split.block}",
                    "창": windowName, "초": elapsed})
    trainedGroups = ",".join(target[-1] for target in run.trained_targets)
    print(f"  inner  {windowName:14s} B{split.block}"
          f"  train {split.train_start:%Y-%m-%d}~{split.train_end:%Y-%m-%d}"
          f"  OOF {split.predict_start:%Y-%m-%d}~{split.predict_end:%Y-%m-%d}"
          f" {len(run.predictions):5d}행  학습G{trainedGroups}  {elapsed:7.1f}s")

timingFrame = pd.DataFrame(timings)
print(f"\n총 학습 {len(timingFrame)}회 · {time.perf_counter() - totalStarted:.1f}s")

### 2-1. 재현 게이트 — 챔피언은 노트북 14·15와 같은 값을 내야 한다

`rf_spatial`의 F0 점수는 노트북 14·15가 `0.5924854297224146`으로 기록했다.
RandomForest는 `random_state`로 트리마다 시드가 정해져 **코어 수와 무관하게 결정적**이므로,
이 값이 재현되지 않으면 원인은 코어가 아니라 패키지 버전이다. 그 경우 아래 모든 숫자를
노트북 14·15와 나란히 놓을 수 없다.

게이트는 **예외를 던지지 않는다.** `reproductionOk`를 표시만 하므로 `False`를 직접 볼 것.

In [ ]:
# 노트북 14·15가 기록한 값. 6자리 반올림이 아니라 계산값 전체다 (설계서 06 6절).
REFERENCE_F0_CHAMPION = 0.5924854297224146
# 노트북 15의 lgbm_pooled − rf_spatial fold 평균. 6자리 기록이라 게이트가 아니라 대조용이다.
REFERENCE_WORKING_DELTA_MEAN = 0.009039

reproduced = float(score_fold(
  trainLabels, predictionCache[(CHAMPION, FOLDS["F0"].train_window)], "F0",
  columns=foldScoreColumns["F0"],
)[0])
gateGap = abs(reproduced - REFERENCE_F0_CHAMPION)
reproductionOk = bool(gateGap <= 1e-9)

print(f"F0 재현 게이트(1e-09) 통과 : {reproductionOk}")
with pd.option_context("display.float_format", lambda value: f"{value:.16g}"):
  display(pd.DataFrame([{
    "후보": CHAMPION, "F0 재현": reproduced,
    "노트북 14·15 기준": REFERENCE_F0_CHAMPION, "절대 차이": gateGap,
    "판정": "통과" if reproductionOk else "실패",
  }]).set_index("후보"))

costFrame = timingFrame.groupby("단계")["초"].agg(["count", "sum", "mean"])
costFrame.columns = ["학습 횟수", "총 초", "평균 초"]
costFrame.loc["합계"] = [costFrame["학습 횟수"].sum(), costFrame["총 초"].sum(), np.nan]
print("학습 비용")
display(costFrame)

---

## 3. 지표 분해 — 노트북 15가 미룬 것

M6를 다음 작업으로 고른 근거는 **개선분이 FICR에서 나온다**는 관찰이다. 노트북 11이
피처 개선분의 79.6%가 FICR 기여라고 쟀고, 노트북 13이 아홉 fold 중 어디에서도 FICR이
주도한다고 확인했다. 그런데 **둘 다 RandomForest로 잰 것**이다.

노트북 15는 모델을 GBM으로 바꾸면서 `one_minus_nmae`와 `ficr`를 `scoreFrame`에 담아
두고도 표시하지 않았다. 그래서 "모델을 바꾼 뒤에도 개선분이 FICR에서 나오는가"는
답이 없는 상태로 남았고, 그 답에 M6의 설계 근거가 걸려 있다.

분해는 산식에서 바로 나온다. `total = 0.5 × (1 − NMAE) + 0.5 × FICR`이므로

> `Δtotal = 0.5 × Δ(1−NMAE) + 0.5 × ΔFICR`

이고, 두 기여의 합이 `Δtotal`과 정확히 같아야 한다. 그 항등식이 성립하는지를 함께 찍는다 —
성립하지 않으면 분해가 아니라 전사 오류다.

`fold 평균 Δtotal`은 노트북 15가 `0.009039`로 기록했다. 6자리 반올림이라 게이트로 쓰지
않고 **대조용**으로만 표시한다.

In [ ]:
championTermsByFold, baseTermsByFold = {}, {}
decomposeRows = []
for foldName in foldOrder:
  spec = FOLDS[foldName]
  columns = foldScoreColumns[foldName]
  championTerms = fold_normalized_terms(
    trainLabels, predictionCache[(CHAMPION, spec.train_window)], foldName, columns
  )
  workingTerms = fold_normalized_terms(
    trainLabels, predictionCache[(WORKING, spec.train_window)], foldName, columns
  )
  championTermsByFold[foldName] = championTerms
  baseTermsByFold[foldName] = workingTerms

  championScore = decompose_normalized(championTerms)
  workingScore = decompose_normalized(workingTerms)
  deltaTotal = workingScore[0] - championScore[0]
  nmaeShare = 0.5 * (workingScore[1] - championScore[1])
  ficrShare = 0.5 * (workingScore[2] - championScore[2])
  decomposeRows.append({
    "fold": foldName, "검증 연도": spec.valid_year, "채점 그룹": len(columns),
    "챔피언 total": championScore[0], "작업모델 total": workingScore[0],
    "Δ total": deltaTotal,
    "Δ 1-NMAE 기여": nmaeShare, "Δ FICR 기여": ficrShare,
    "FICR 기여율": ficrShare / deltaTotal if deltaTotal else np.nan,
  })

decomposeFrame = pd.DataFrame(decomposeRows).set_index("fold")
print(f"{WORKING} vs 챔피언({CHAMPION}) — Δ를 두 지표 기여로 쪼갠다")
display(decomposeFrame)

rebuildGap = float(
  (decomposeFrame["Δ 1-NMAE 기여"] + decomposeFrame["Δ FICR 기여"] - decomposeFrame["Δ total"])
  .abs().max()
)
meanDelta = float(decomposeFrame["Δ total"].mean())
ficrLedFolds = int((decomposeFrame["FICR 기여율"] > 0.5).sum())
print(f"\n분해 항등식 최대 오차   : {rebuildGap:.2e}  (0.5·Δ(1-NMAE) + 0.5·ΔFICR = Δtotal)")
print(f"fold 평균 Δ total       : {meanDelta:.6f}"
      f"   노트북 15 기록 {REFERENCE_WORKING_DELTA_MEAN:.6f}"
      f"   차이 {meanDelta - REFERENCE_WORKING_DELTA_MEAN:+.6f}")
print(f"Δ total 부호 + fold     : {int((decomposeFrame['Δ total'] > 0).sum())}/{len(foldOrder)}")
print(f"FICR 기여율 중앙값      : {decomposeFrame['FICR 기여율'].median():.1%}")
print(f"FICR 기여율 > 50% fold  : {ficrLedFolds}/{len(foldOrder)}")
print(f"1-NMAE 기여가 음수인 fold: {int((decomposeFrame['Δ 1-NMAE 기여'] < 0).sum())}/{len(foldOrder)}")
print()
print("=== M6 전제 판정 ===")
print(f"  노트북 11(RF·2024 holdout) FICR 기여율 : 79.6%")
print(f"  이 노트북({WORKING}·아홉 fold) 중앙값   : {decomposeFrame['FICR 기여율'].median():.1%}")
print(f"  FICR이 개선분을 주도하는 fold           : {ficrLedFolds}/{len(foldOrder)}")
print(f"  전제 유지 여부                          :"
      f" {'유지 — 경계를 겨냥할 근거가 남아 있다' if ficrLedFolds > len(foldOrder) / 2 else '흔들림 — 5절 오라클 상한을 근거로 다시 판단할 것'}")

숫자를 그림으로 옮긴다. 왼쪽이 두 기여의 크기, 오른쪽이 FICR 기여율이다.

In [ ]:
yearColor = {2022: "#c44e52", 2023: "#dd8452", 2024: "#4c72b0"}
foldTickLabels = [f"{name}\n{FOLDS[name].valid_year}" for name in foldOrder]
positions = np.arange(len(foldOrder))
yearSpan = {}
for name in foldOrder:
  yearSpan.setdefault(FOLDS[name].valid_year, []).append(foldOrder.index(name))

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.4), gridspec_kw={"width_ratios": [1.3, 1]})
for axis in axes:
  for year, spanPositions in yearSpan.items():
    axis.axvspan(min(spanPositions) - 0.5, max(spanPositions) + 0.5,
                 color=yearColor[year], alpha=0.07, zorder=0)

width = 0.3
axes[0].bar(positions - width / 2, decomposeFrame["Δ 1-NMAE 기여"], width,
            label="0.5 x Δ(1-NMAE)", color="#4c72b0", zorder=3)
axes[0].bar(positions + width / 2, decomposeFrame["Δ FICR 기여"], width,
            label="0.5 x ΔFICR", color="#dc2626", zorder=3)
axes[0].plot(positions, decomposeFrame["Δ total"], marker="o", color="#333333",
             linewidth=1.6, label="Δ total", zorder=5)
axes[0].axhline(0, color="#333333", linewidth=1, zorder=4)
axes[0].axhline(RANK_THRESHOLD, color="#7f7f7f", linestyle="--", linewidth=1.2, zorder=4,
                label=f"순위 문턱 {RANK_THRESHOLD}")
axes[0].set_title(f"{WORKING} 에서 {CHAMPION} 을 뺀 값을 지표 기여로 쪼갠 것", fontsize=11.5)
axes[0].set_ylabel("total_score 기여")
# 범례가 막대를 덮지 않도록 위쪽에 여유를 만든다.
decomposeSpan = float(decomposeFrame[["Δ 1-NMAE 기여", "Δ FICR 기여", "Δ total"]].to_numpy().max())
decomposeFloor = float(decomposeFrame[["Δ 1-NMAE 기여", "Δ FICR 기여", "Δ total"]].to_numpy().min())
axes[0].set_ylim(min(decomposeFloor * 1.25, -0.0005), max(decomposeSpan * 1.42, RANK_THRESHOLD * 1.6))
axes[0].legend(fontsize=9, ncol=2, loc="upper left")

axes[1].bar(positions, decomposeFrame["FICR 기여율"],
            color=[yearColor[FOLDS[name].valid_year] for name in foldOrder], zorder=3)
axes[1].axhline(0.5, color="#333333", linestyle="--", linewidth=1.4, zorder=4,
                label="0.5 — 두 지표가 반반")
axes[1].axhline(0.796, color="#c44e52", linestyle=":", linewidth=1.6, zorder=4,
                label="0.796 — 노트북 11 (RF)")
axes[1].set_title("개선분 중 FICR 기여의 비중", fontsize=11.5)
axes[1].set_ylabel("FICR 기여율")
axes[1].legend(fontsize=9)

for axis in axes:
  axis.set_xlabel("fold · 검증 연도")
  axis.set_xticks(positions)
  axis.set_xticklabels(foldTickLabels)
fig.suptitle("그림 1. 모델을 GBM으로 바꾼 뒤에도 개선분이 FICR에서 오는가", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

---

## 4. 오차율이 계단 경계에 대해 어떻게 놓여 있나

FICR이 지렛대라면 그 지렛대의 길이는 **경계 근처에 몇 행이 몰려 있는가**로 정해진다.
노트북 12가 제출 S2에서 이 값을 18.5%(6%·8% 경계 ±1%p)로 쟀는데, 그것은 2025 제출
구간의 값이다. 여기서는 아홉 fold의 검증 구간에서 다시 잰다.

밴드별 비율도 함께 본다. `4원` 비율이 이미 매우 높다면 남은 여지는 작고, `0원` 비율이
크다면 경계를 한 칸 옮겨 얻을 것이 많다. 어느 쪽인지가 5절 오라클 상한의 크기를 결정한다.

In [ ]:
densityRows = []
for foldName in foldOrder:
  frame = boundary_density(baseTermsByFold[foldName], window=0.01)
  weight = frame["평가 행"] / frame["평가 행"].sum()
  densityRows.append({
    "fold": foldName, "검증 연도": FOLDS[foldName].valid_year,
    "평가 행": int(frame["평가 행"].sum()),
    "4원 비율": float((frame["4원 비율"] * weight).sum()),
    "3원 비율": float((frame["3원 비율"] * weight).sum()),
    "0원 비율": float((frame["0원 비율"] * weight).sum()),
    "경계 ±1%p 비율": float((frame["경계 ±1% 비율"] * weight).sum()),
    "평균 오차율": float((frame["평균 오차율"] * weight).sum()),
  })
densityFrame = pd.DataFrame(densityRows).set_index("fold")
print(f"{WORKING}의 오차율 분포 — 평가 행 가중 평균 (그룹 수가 fold마다 다르다)")
display(densityFrame)

print(f"\n경계 ±1%p 비율 범위     : {densityFrame['경계 ±1%p 비율'].min():.1%}"
      f" ~ {densityFrame['경계 ±1%p 비율'].max():.1%}"
      f"  (노트북 12의 S2 제출 구간 18.5%)")
print(f"4원 비율 범위           : {densityFrame['4원 비율'].min():.1%} ~ {densityFrame['4원 비율'].max():.1%}")
print(f"0원 비율 범위           : {densityFrame['0원 비율'].min():.1%} ~ {densityFrame['0원 비율'].max():.1%}")
print()
print("그룹별로도 본다 — F0 (2024 전체 · 3그룹)")
display(boundary_density(baseTermsByFold["F0"], window=0.01))

분포를 그림으로 본다. 왼쪽은 F0의 오차율 히스토그램과 두 경계, 오른쪽은 fold마다
경계 ±1%p에 몰린 비율이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2), gridspec_kw={"width_ratios": [1.25, 1]})

f0ErrorRates = np.concatenate([
  np.abs(predicted - actual) for predicted, actual in baseTermsByFold["F0"].values() if actual.size
])
axes[0].hist(f0ErrorRates, bins=90, range=(0.0, 0.30), color="#4c72b0", zorder=3)
for edge, color, text in [(0.06, "#c44e52", "6% — 4원 경계"), (0.08, "#dd8452", "8% — 3원 경계")]:
  axes[0].axvline(edge, color=color, linestyle="--", linewidth=1.8, zorder=4, label=text)
axes[0].set_title(f"F0 오차율 분포 — {WORKING} · 평가 행 {f0ErrorRates.size:,}개", fontsize=11.5)
axes[0].set_xlabel("오차율 = |예측 - 실측| / 설비용량")
axes[0].set_ylabel("행 수")
axes[0].legend(fontsize=9.5)

bandBottom = np.zeros(len(foldOrder))
for column, color in [("4원 비율", "#55a868"), ("3원 비율", "#dd8452"), ("0원 비율", "#c44e52")]:
  values = densityFrame[column].to_numpy()
  axes[1].bar(positions, values, 0.62, bottom=bandBottom, label=column, color=color, zorder=3)
  bandBottom = bandBottom + values
axes[1].plot(positions, densityFrame["경계 ±1%p 비율"], marker="o", color="#111111",
             linewidth=1.8, label="경계 ±1%p 비율", zorder=5)
axes[1].axhline(0.185, color="#111111", linestyle=":", linewidth=1.4, zorder=4,
                label="0.185 — 노트북 12 S2")
axes[1].set_ylim(0, 1.45)
axes[1].set_title("단가 밴드 구성과 경계 근처 밀도", fontsize=11.5)
axes[1].set_ylabel("평가 행 비율")
axes[1].set_xlabel("fold · 검증 연도")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(foldTickLabels)
axes[1].legend(fontsize=8.5, ncol=2, loc="upper center")

fig.suptitle("그림 2. FICR은 계단 함수이고, 지렛대의 길이는 경계 근처 밀도가 정한다", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

---

## 5. 오라클 상한 — 천장을 먼저 재고 후보를 믿는다

보정 후보는 파라미터가 1~5개뿐이다. Δ가 0에 가깝게 나올 가능성이 높고, 그때
**"보정이라는 방향 자체가 무력한 것"**과 **"우리 후보 6종이 약한 것"**을 구분할 수 없으면
어느 쪽으로도 결론을 낼 수 없다.

그래서 상한을 먼저 잰다. 행마다 **이상적인 `±δ` 밀기**를 허용했을 때의 점수다.
두 지표 모두 오차율에 대해 단조이므로(NMAE는 증가, 단가는 비증가) 행별 최적은
오차를 `δ`만큼 0으로 당기는 것이고 그 사이에 상충이 없다. 따라서 이 값은
**행별 보정 크기가 `δ` 이하인 어떤 보정도 넘을 수 없는 천장**이다.

달성 가능한 값이 아니다. 행마다 실측을 보고 방향을 정하는 것이므로 예측 시점에는
불가능하다. 쓰는 방법은 하나다 — **상한이 순위 문턱 `0.0036`보다 작으면 이 방향에
남은 것이 없다.** 그 경우 6절 이하의 후보 성적은 볼 필요 없이 M6는 기각이다.

In [ ]:
ORACLE_DELTAS = [0.0025, 0.005, 0.01, 0.02, 0.04]

oracleRows = []
for foldName in foldOrder:
  terms = baseTermsByFold[foldName]
  base = score_normalized(terms)
  row = {"fold": foldName, "검증 연도": FOLDS[foldName].valid_year, "보정 없음": base}
  for delta in ORACLE_DELTAS:
    row[f"δ={delta:g}"] = oracle_row_bound(terms, delta) - base
  oracleRows.append(row)
oracleFrame = pd.DataFrame(oracleRows).set_index("fold")
print("행별 이상 보정의 상한 — 보정 없음 대비 Δ (달성 불가능한 천장)")
display(oracleFrame)

deltaColumns = [column for column in oracleFrame.columns if column.startswith("δ=")]
print(f"\n순위 문턱 {RANK_THRESHOLD} 와 비교 (fold 평균 기준)")
smallestUseful = None
for column in deltaColumns:
  mean = float(oracleFrame[column].mean())
  minimum = float(oracleFrame[column].min())
  crosses = mean > RANK_THRESHOLD
  if crosses and smallestUseful is None:
    smallestUseful = column
  print(f"  {column:10s} 상한 Δ 평균 {mean:.6f} (문턱의 {mean / RANK_THRESHOLD:5.2f}배)"
        f"  최소 {minimum:.6f}  {'문턱 초과' if crosses else '문턱 미달'}")

print()
print("=== 방향 판정 ===")
if smallestUseful is None:
  print(f"  어떤 δ에서도 상한이 문턱을 넘지 못한다 → 보정 방향 자체에 남은 것이 없다")
else:
  print(f"  상한이 문턱을 넘는 가장 작은 보정 크기 : {smallestUseful}")
  print(f"  즉 이 크기 이상으로 움직이는 보정이라면 원리상 문턱을 넘을 여지가 있다")
print(f"  주의 : 상한은 행마다 실측을 보고 방향을 정한 값이다. 실제 후보는 예측만 보고")
print(f"         한 방향으로 움직이므로 이 값의 일부만 가져온다. 그 간격을 9절에서 쪼갠다")

`δ`를 키우면 상한은 단조로 올라간다. 어디서 문턱을 넘는지가 이 그림의 요점이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2), gridspec_kw={"width_ratios": [1.15, 1]})

# 색은 검증 연도, 마커는 fold다. 연도만으로 칠하면 2024 fold 다섯 개가 구분되지 않는다.
foldMarkers = ["o", "s", "^", "v", "D", "P", "X", "*", "<"]
for order, foldName in enumerate(foldOrder):
  axes[0].plot([0.0] + ORACLE_DELTAS,
               [0.0] + [float(oracleFrame.loc[foldName, f"δ={delta:g}"]) for delta in ORACLE_DELTAS],
               marker=foldMarkers[order % len(foldMarkers)], markersize=5.5, linewidth=1.4,
               label=f"{foldName} ({FOLDS[foldName].valid_year})",
               color=yearColor[FOLDS[foldName].valid_year], alpha=0.8, zorder=3)
axes[0].axhline(RANK_THRESHOLD, color="#111111", linestyle="--", linewidth=1.6, zorder=4,
                label=f"순위 문턱 {RANK_THRESHOLD}")
axes[0].set_title("보정 크기 δ 를 키우면 천장이 어디까지 올라가나", fontsize=11.5)
axes[0].set_xlabel("허용한 행별 보정 크기 δ (설비용량 비율)")
axes[0].set_ylabel("보정 없음 대비 상한 Δ")
axes[0].legend(fontsize=8, ncol=2, loc="upper left")

meanBounds = [float(oracleFrame[f"δ={delta:g}"].mean()) for delta in ORACLE_DELTAS]
axes[1].bar([f"δ={delta:g}" for delta in ORACLE_DELTAS], meanBounds, color="#4c72b0", zorder=3)
axes[1].axhline(RANK_THRESHOLD, color="#c44e52", linestyle="--", linewidth=1.6, zorder=4,
                label=f"순위 문턱 {RANK_THRESHOLD}")
axes[1].set_title("fold 평균 상한 Δ", fontsize=11.5)
axes[1].set_ylabel("보정 없음 대비 상한 Δ")
axes[1].legend(fontsize=9.5)

fig.suptitle("그림 3. 오라클 상한 — 이 방향에 남은 여지의 천장", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

---

## 6. 보정을 fit하고 아홉 fold에서 채점한다

fit은 학습 창 내부 OOF에서만 한다(Decision Box ㉗). 창 하나에서 두 가지 범위로 fit한다 —
**3블록 전체**와 **마지막 블록만**이다. 후자는 8절의 민감도 확인용이며 추가 학습이 없다.

같은 학습 창을 쓰는 fold는 같은 보정을 받는다. 학습을 공유하니 보정도 공유하는 것이
맞다 — F0·F1·F2가 `W_2022_2023` 하나를 쓰고, F3·F8이 `W_2023`을 쓴다.

fit 결과에는 `OOF Δ`도 함께 싣는다. **fit한 그 데이터에서의 개선분**이므로 표본 밖 성능이
아니다. `b = 0`·`a = 1`이 전부 그리드 안에 있으니 이 값은 음수가 될 수 없고, 크기는
"이 함수 형태가 OOF에서 얼마나 당길 수 있었나"만 말한다.

In [ ]:
OOF_VARIANTS = ["3블록", "마지막 블록"]


def oofTermsFor(windowName, variant):
  blocks = [split.block for split in oofSplits[windowName]]
  if variant == "마지막 블록":
    blocks = blocks[-1:]
  frame = pd.concat([oofPredictions[(windowName, block)] for block in blocks])
  # 라벨이 없는 그룹은 평가 대상 마스크에서 자동으로 빠진다 (W_2022의 Group 3).
  return normalized_terms(trainLabels, frame, frame.index, TARGET_COLS)


fittedCalibrations, fitRows = {}, []
fitStarted = time.perf_counter()
for windowName in windowOrder:
  for variant in OOF_VARIANTS:
    terms = oofTermsFor(windowName, variant)
    oofRows = {column: int(pair[0].size) for column, pair in terms.items()}
    baseScore = score_normalized(terms)
    for name in CALIBRATORS:
      calibration = fit_calibration(name, terms)
      fittedCalibrations[(windowName, variant, name)] = calibration
      fitRows.append({
        "창": windowName, "OOF 범위": variant, "후보": name,
        "OOF 평가 행": sum(oofRows.values()),
        "OOF G1/G2/G3": "/".join(str(oofRows[column]) for column in TARGET_COLS),
        "파라미터": calibration.describe(),
        "OOF Δ (fit 데이터)": score_normalized(terms, calibration) - baseScore,
      })
fitFrame = pd.DataFrame(fitRows)
print(f"보정 fit {len(fitFrame)}회 · {time.perf_counter() - fitStarted:.1f}s")

print("\nOOF 3블록 전체로 fit한 결과")
display(
  fitFrame[fitFrame["OOF 범위"] == "3블록"]
  .set_index(["창", "후보"])[["OOF 평가 행", "OOF G1/G2/G3", "파라미터", "OOF Δ (fit 데이터)"]]
)
print("주의 : 위 Δ는 fit한 그 데이터에서의 값이다. 표본 밖 성적은 아래에서 잰다")

### 6-1. fold 채점과 판정

판정 규칙은 설계서 06 5.4절이며 노트북 15 Decision Box ㉕가 고친 판본을 쓴다.

| 상황 | 판정 |
|------|------|
| 간격 > 0.0036 **이고** 검증 연도를 덮는 fold 전부에서 부호 일치 | 순위 확정 |
| 간격 ≤ 0.0036 이지만 검증 연도 둘 이상을 덮는 fold 전부에서 부호 일치 | 잠정 채택 |
| 부호가 흔들림 (**크기와 무관**) | 순위 미확정 — 더 단순한 후보 유지 |

`판정 (2024만)` 열을 나란히 둔다. 노트북 14가 보인 실패 모드 — 한 해만 검증하는 fold
다섯 개가 "부호 전부 일치"라는 가짜 확신을 발급한다 — 가 보정에서도 재현되는지 보기
위해서다. 노트북 15에서는 그 조합이 실제로 나왔다.

In [ ]:
scoreRows = []
for foldName in foldOrder:
  spec = FOLDS[foldName]
  terms = baseTermsByFold[foldName]
  for name in CALIBRATORS:
    calibration = fittedCalibrations[(spec.train_window, "3블록", name)]
    total, oneMinusNmae, ficr = decompose_normalized(terms, calibration)
    scoreRows.append({
      "fold": foldName, "후보": name, "검증 연도": spec.valid_year,
      "total_score": total, "one_minus_nmae": oneMinusNmae, "ficr": ficr,
    })
scoreFrame = pd.DataFrame(scoreRows)
totalPivot = scoreFrame.pivot(index="후보", columns="fold", values="total_score").loc[
  list(CALIBRATORS), foldOrder
]
ficrPivot = scoreFrame.pivot(index="후보", columns="fold", values="ficr").loc[
  list(CALIBRATORS), foldOrder
]
deltaVsControl = totalPivot - totalPivot.loc[CONTROL]
folds2024 = [name for name in foldOrder if FOLDS[name].valid_year == 2024]


def judge(delta, folds):
  # 크기와 부호를 함께 본다. 둘 중 하나만으로는 확정하지 않는다 (노트북 15와 같은 함수).
  values = delta[folds]
  positive, count = int((values > 0).sum()), len(values)
  meanGap = float(values.mean())
  if positive == count and meanGap > RANK_THRESHOLD:
    return meanGap, positive, count, "우위 확정"
  if positive == 0 and abs(meanGap) > RANK_THRESHOLD:
    return meanGap, positive, count, "열위 확정"
  if positive == count:
    return meanGap, positive, count, "잠정 우위 (부호 일치·크기 미달)"
  return meanGap, positive, count, "미확정 (부호 흔들림)"


print("fold별 total_score (보정 후보 × fold)")
display(totalPivot)
print(f"통제군({CONTROL}) 대비 Δ")
display(deltaVsControl.drop(index=[CONTROL]))

judgeRows = []
for name in CALIBRATORS:
  if name == CONTROL:
    continue
  delta = deltaVsControl.loc[name]
  meanGap, positive, count, verdict = judge(delta, foldOrder)
  _, positive2024, count2024, verdict2024 = judge(delta, folds2024)
  judgeRows.append({
    "후보": name, "Δ 평균": meanGap, "Δ 최소": float(delta.min()), "Δ 최대": float(delta.max()),
    "문턱 대비": meanGap / RANK_THRESHOLD,
    "부호 + (9 fold)": f"{positive}/{count}", "판정 (9 fold)": verdict,
    "부호 + (2024만)": f"{positive2024}/{count2024}", "판정 (2024만)": verdict2024,
  })
judgeFrame = pd.DataFrame(judgeRows).set_index("후보")
print(f"판정 — 순위 문턱 {RANK_THRESHOLD}")
display(judgeFrame)

robustCandidates = [name for name in judgeFrame.index if judgeFrame.loc[name, "판정 (9 fold)"] == "우위 확정"]
tentativeCandidates = [
  name for name in judgeFrame.index
  if judgeFrame.loc[name, "판정 (9 fold)"] == "잠정 우위 (부호 일치·크기 미달)"
]
bestCandidate = deltaVsControl.drop(index=[CONTROL]).mean(axis=1).idxmax()

print(f"\n우위 확정 후보 : {robustCandidates or '없음'}")
print(f"잠정 우위 후보 : {tentativeCandidates or '없음'}")
print(f"Δ 평균 최고    : {bestCandidate} ({float(deltaVsControl.loc[bestCandidate].mean()):+.6f})")
print()
print("=== 2024 fold만 봤다면 어떤 결론이 나왔을까 ===")
for name in judgeFrame.index:
  negativeFolds = [fold for fold in foldOrder if deltaVsControl.loc[name, fold] <= 0]
  print(f"  {name:16s} 2024만 {judgeFrame.loc[name, '부호 + (2024만)']}"
        f" → {judgeFrame.loc[name, '판정 (2024만)']:30s}"
        f" | 9 fold {judgeFrame.loc[name, '부호 + (9 fold)']}"
        f" → {judgeFrame.loc[name, '판정 (9 fold)']}"
        f"{'  음수 fold ' + ','.join(negativeFolds) if negativeFolds else ''}")

Δ를 fold별로 그린다. 문턱선 위로 올라가면서 0 아래로 내려가지 않는 후보만 첫째 칸이다.

In [ ]:
candidateNames = [name for name in CALIBRATORS if name != CONTROL]
candidateColor = dict(zip(candidateNames, ["#8c8c8c", "#4c72b0", "#55a868", "#dd8452", "#c44e52"]))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.4), gridspec_kw={"width_ratios": [1.35, 1]})
for year, spanPositions in yearSpan.items():
  axes[0].axvspan(min(spanPositions) - 0.5, max(spanPositions) + 0.5,
                  color=yearColor[year], alpha=0.07, zorder=0)

width = 0.16
for offset, name in zip(np.linspace(-2, 2, len(candidateNames)) * width, candidateNames):
  axes[0].bar(positions + offset, [deltaVsControl.loc[name, fold] for fold in foldOrder],
              width, label=name, color=candidateColor[name], zorder=3)
axes[0].axhline(0, color="#333333", linewidth=1, zorder=4)
axes[0].axhline(RANK_THRESHOLD, color="#111111", linestyle="--", linewidth=1.4, zorder=4,
                label=f"순위 문턱 {RANK_THRESHOLD}")
axes[0].set_title(f"통제군({CONTROL}) 대비 Δ — 보정 후보 5종 × 아홉 fold", fontsize=11.5)
axes[0].set_ylabel("total_score 차이")
axes[0].set_xlabel("fold · 검증 연도")
axes[0].set_xticks(positions)
axes[0].set_xticklabels(foldTickLabels)
# 범례가 막대 위에 앉도록 위쪽 여유를 만든다. 음수 Δ도 잘리지 않게 아래도 함께 넓힌다.
deltaValues = deltaVsControl.drop(index=[CONTROL]).to_numpy()
deltaTop, deltaFloor = float(deltaValues.max()), float(deltaValues.min())
axes[0].set_ylim(min(deltaFloor * 1.3, -RANK_THRESHOLD * 0.4),
                 max(deltaTop * 1.45, RANK_THRESHOLD * 1.8))
axes[0].legend(fontsize=8.5, ncol=3, loc="upper left")

meanGaps = [float(deltaVsControl.loc[name].mean()) for name in candidateNames]
signCounts = [int((deltaVsControl.loc[name] > 0).sum()) for name in candidateNames]
bars = axes[1].bar(candidateNames, meanGaps,
                   color=[candidateColor[name] for name in candidateNames], zorder=3)
axes[1].axhline(RANK_THRESHOLD, color="#111111", linestyle="--", linewidth=1.4, zorder=4,
                label=f"순위 문턱 {RANK_THRESHOLD}")
axes[1].axhline(0, color="#333333", linewidth=1, zorder=4)
for bar, count in zip(bars, signCounts):
  height = bar.get_height()
  axes[1].annotate(f"부호 {count}/{len(foldOrder)}",
                   (bar.get_x() + bar.get_width() / 2, height),
                   textcoords="offset points", xytext=(0, 5 if height >= 0 else -12),
                   ha="center", fontsize=8.5)
axes[1].set_title("fold 평균 Δ 와 부호 일치 수", fontsize=11.5)
axes[1].set_ylabel("fold 평균 Δ")
axes[1].tick_params(axis="x", labelrotation=20)
# 막대 위 주석이 축 밖으로 나가지 않게 여유를 준다.
meanTop, meanFloor = max(meanGaps), min(meanGaps)
axes[1].set_ylim(min(meanFloor * 1.4, -RANK_THRESHOLD * 0.5),
                 max(meanTop * 1.32, RANK_THRESHOLD * 1.8))
axes[1].legend(fontsize=9.5, loc="upper left")

fig.suptitle("그림 4. 보정 후보의 아홉 fold 성적 — 크기와 부호를 함께 본다", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

### 6-2. fold 내부 bootstrap — 이 Δ가 표본 잡음과 구분되는가

fold 간 SD 0.0036은 **검증 구간과 학습 창이 바뀔 때**의 변동이다. 그것과 별개로,
같은 fold 안에서 평가 행만 재표집했을 때 Δ의 부호가 유지되는지를 봐야 한다.
노트북 12·15와 같은 paired bootstrap이며, 같은 재표집 인덱스를 두 예측에 적용하므로
표본이 만드는 공통 변동은 상쇄되고 두 예측의 차이만 남는다.

여기서 **프레임 경로도 함께 검증한다.** 위 채점은 정규화 공간의 `terms`에서 했고,
실제 제출은 `apply_calibration()`이 원단위 예측 프레임을 고친 결과를 쓴다. 두 경로가
같은 Δ를 내지 않으면 이 노트북의 숫자와 제출물이 다른 것을 가리킨다.

In [ ]:
BOOTSTRAP_DRAWS = 4000

bootRows = []
bootStarted = time.perf_counter()
for foldName in foldOrder:
  spec = FOLDS[foldName]
  columns = foldScoreColumns[foldName]
  rawPredictions = predictionCache[(WORKING, spec.train_window)]
  calibration = fittedCalibrations[(spec.train_window, "3블록", bestCandidate)]
  calibratedPredictions = apply_calibration(rawPredictions, calibration, columns=columns)

  calibratedTerms = group_error_terms(trainLabels, calibratedPredictions, foldName, columns)
  rawTerms = group_error_terms(trainLabels, rawPredictions, foldName, columns)
  gaps = paired_bootstrap_gap(calibratedTerms, rawTerms, draws=BOOTSTRAP_DRAWS)

  frameDelta = total_from_terms(calibratedTerms) - total_from_terms(rawTerms)
  termsDelta = float(deltaVsControl.loc[bestCandidate, foldName])
  bootRows.append({
    "fold": foldName, "검증 연도": spec.valid_year,
    f"{bestCandidate}−{CONTROL}": termsDelta,
    "P(보정 > 무보정)": float((gaps > 0).mean()),
    "bootstrap Δ 중앙값": float(np.median(gaps)),
    "프레임 경로 Δ": frameDelta,
    "경로 차이": abs(frameDelta - termsDelta),
  })
bootstrapFrame = pd.DataFrame(bootRows).set_index("fold")
print(f"Δ 평균 최고 후보 : {bestCandidate}")
display(bootstrapFrame)
print(f"bootstrap {BOOTSTRAP_DRAWS}회 × {len(foldOrder)} fold · {time.perf_counter() - bootStarted:.1f}s")

pathOk = bool(float(bootstrapFrame["경로 차이"].max()) <= 1e-9)
print(f"\nterms 경로와 프레임 경로가 일치하는가(1e-09) : {pathOk}"
      f"   최대 차이 {float(bootstrapFrame['경로 차이'].max()):.2e}")
print(f"P(보정 > 무보정) 범위 : {bootstrapFrame['P(보정 > 무보정)'].min():.3f}"
      f" ~ {bootstrapFrame['P(보정 > 무보정)'].max():.3f}")
print(f"P > 0.95 인 fold      : {int((bootstrapFrame['P(보정 > 무보정)'] > 0.95).sum())}/{len(foldOrder)}")
print(f"P가 0.4~0.6 인 fold   :"
      f" {int(((bootstrapFrame['P(보정 > 무보정)'] > 0.4) & (bootstrapFrame['P(보정 > 무보정)'] < 0.6)).sum())}"
      f"/{len(foldOrder)}  (동전 던지기와 구분되지 않는 구간)")

---

## 7. 목적함수를 바꾸면 정말 다른 보정이 나오는가

이 절이 "metric-aware"라는 말의 값을 정한다. `bias_mae`와 `bias_metric`은 함수 형태가
같고(`p̂ + b`), 그리드가 같고(`±0.06` step `0.001`), fit 데이터가 같다. 다른 것은
목적함수 하나뿐이다.

**둘이 같은 `b`를 고른다면** 이 데이터에서는 계단 구조가 최적 보정을 바꾸지 못한다는
뜻이며, M6의 "metric-aware" 부분은 이름뿐이 된다. 그것도 결과다 — 그 경우 남는 것은
"보정 자체가 도움이 되는가"라는 더 좁은 질문이다.

파라미터가 학습 창마다 얼마나 흔들리는지도 함께 본다. 창이 바뀔 때 `b`의 **부호까지**
바뀐다면, 그 보정은 데이터의 안정적인 성질이 아니라 창마다 다른 잡음을 맞추고 있는 것이다.

In [ ]:
biasFrame = (
  fitFrame[(fitFrame["OOF 범위"] == "3블록") & (fitFrame["후보"].isin(["bias_mae", "bias_metric"]))]
  .pivot(index="창", columns="후보", values="파라미터")
  .loc[windowOrder]
)
biasValues = pd.DataFrame(
  {
    name: [
      float(fittedCalibrations[(windowName, "3블록", name)].params["b"])
      for windowName in windowOrder
    ]
    for name in ["bias_mae", "bias_metric"]
  },
  index=windowOrder,
)
biasValues["차이 (metric − mae)"] = biasValues["bias_metric"] - biasValues["bias_mae"]
print("단일 가산 보정 b — 목적함수만 다르다")
display(biasValues)

affineValues = pd.DataFrame(
  {
    "a": [float(fittedCalibrations[(w, "3블록", "affine_metric")].params["a"]) for w in windowOrder],
    "b": [float(fittedCalibrations[(w, "3블록", "affine_metric")].params["b"]) for w in windowOrder],
  },
  index=windowOrder,
)
print("affine 보정 파라미터")
display(affineValues)

identicalWindows = int((biasValues["차이 (metric − mae)"].abs() <= 1e-12).sum())
signFlips = int(
  (np.sign(biasValues["bias_metric"]) != np.sign(biasValues["bias_metric"].iloc[0])).sum()
)
print(f"\n두 목적함수가 같은 b를 고른 창 : {identicalWindows}/{len(windowOrder)}")
print(f"bias_metric의 b 범위          : {biasValues['bias_metric'].min():+.4f}"
      f" ~ {biasValues['bias_metric'].max():+.4f}")
print(f"bias_metric의 b 부호가 첫 창과 다른 창 : {signFlips}/{len(windowOrder)}")
print(f"affine a 범위                 : {affineValues['a'].min():.2f} ~ {affineValues['a'].max():.2f}"
      f"   (1.0에서 벗어난 창 {int((affineValues['a'] != 1.0).sum())}/{len(windowOrder)})")
print()
print("=== metric-aware 판정 ===")
if identicalWindows == len(windowOrder):
  print("  모든 창에서 두 목적함수가 같은 b를 골랐다 → 이 데이터에서 metric-aware는 이름뿐이다")
else:
  print(f"  {len(windowOrder) - identicalWindows}개 창에서 갈렸다 → 목적함수가 결과를 바꾼다")
  print(f"  갈린 창의 판정은 6절 표의 bias_mae / bias_metric 행을 대조할 것")

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.0), gridspec_kw={"width_ratios": [1.15, 1]})
windowPositions = np.arange(len(windowOrder))
barWidth = 0.36
axes[0].bar(windowPositions - barWidth / 2, biasValues["bias_mae"], barWidth,
            label="bias_mae (NMAE 최소)", color="#8c8c8c", zorder=3)
axes[0].bar(windowPositions + barWidth / 2, biasValues["bias_metric"], barWidth,
            label="bias_metric (total_score 최대)", color="#c44e52", zorder=3)
axes[0].axhline(0, color="#333333", linewidth=1, zorder=4)
axes[0].set_title("같은 함수 형태·같은 그리드, 목적함수만 다른 b", fontsize=11.5)
axes[0].set_ylabel("가산 보정 b (설비용량 비율)")
axes[0].set_xticks(windowPositions)
axes[0].set_xticklabels(windowOrder, rotation=20, ha="right")
# b는 부호가 어느 쪽으로도 갈 수 있다. 0을 항상 포함하고 양쪽에 여유를 둬 막대가 잘리지 않게 한다.
biasSpan = float(np.abs(biasValues[["bias_mae", "bias_metric"]].to_numpy()).max())
biasSpan = max(biasSpan, 1e-3)
axes[0].set_ylim(-biasSpan * 1.55, biasSpan * 1.55)
axes[0].legend(fontsize=9, loc="upper left" if biasValues[["bias_mae", "bias_metric"]].to_numpy().mean() < 0 else "lower left")

axes[1].plot(windowPositions, affineValues["a"], marker="o", color="#4c72b0",
             linewidth=1.8, label="기울기 a", zorder=3)
axes[1].axhline(1.0, color="#333333", linestyle="--", linewidth=1.2, zorder=4, label="a = 1 (변화 없음)")
secondary = axes[1].twinx()
secondary.plot(windowPositions, affineValues["b"], marker="s", color="#dd8452",
               linewidth=1.8, label="절편 b", zorder=3)
secondary.grid(False)
secondary.set_ylabel("절편 b")
axes[1].set_title("affine 보정 파라미터의 창별 변동", fontsize=11.5)
axes[1].set_ylabel("기울기 a")
axes[1].set_xticks(windowPositions)
axes[1].set_xticklabels(windowOrder, rotation=20, ha="right")
# 두 축의 선과 a=1 기준선을 범례가 가리지 않도록 위쪽을 넓힌다.
slopeSpan = max(float(np.abs(affineValues["a"] - 1.0).max()), 0.01)
axes[1].set_ylim(1.0 - slopeSpan * 1.4, 1.0 + slopeSpan * 2.4)
handles, labelTexts = axes[1].get_legend_handles_labels()
extraHandles, extraLabels = secondary.get_legend_handles_labels()
axes[1].legend(handles + extraHandles, labelTexts + extraLabels, fontsize=9,
               loc="upper center", ncol=3)

fig.suptitle("그림 5. 목적함수와 학습 창이 보정 파라미터를 얼마나 움직이나", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

---

## 8. OOF 분할이 결론을 만들지 않았는지 확인한다

Decision Box ㉗은 블록 수를 3으로 고정했다. 그 선택이 판정을 만들었다면 판정은
데이터의 성질이 아니라 우리가 고른 분할의 성질이다.

추가 학습 없이 확인할 수 있다. 같은 OOF 예측을 두 범위로 나눠 다시 fit한다.

| 범위 | inner 학습 크기 | OOF 표본 |
|------|-----------------|----------|
| 3블록 전체 | 창의 25% · 50% · 75% | 창의 75% |
| 마지막 블록만 | 창의 75% | 창의 25% |

**마지막 블록만 쓰는 쪽이 최종 모델과 가장 잘 맞는 조각이다** — inner 학습이 창의 75%로
최종 모델(100%)에 가장 가깝다. 대신 OOF 표본은 1/3이다. 3블록은 그 반대다. 둘의 편향이
반대 방향이므로, **두 판정이 같다면 이 축이 결론을 결정하지 않았다**고 말할 수 있다.

In [ ]:
sensitivityRows = []
sensitivityDeltas = {}
for variant in OOF_VARIANTS:
  variantRows = []
  for foldName in foldOrder:
    spec = FOLDS[foldName]
    for name in CALIBRATORS:
      total = decompose_normalized(
        baseTermsByFold[foldName], fittedCalibrations[(spec.train_window, variant, name)]
      )[0]
      variantRows.append({"fold": foldName, "후보": name, "total_score": total})
  variantPivot = (
    pd.DataFrame(variantRows)
    .pivot(index="후보", columns="fold", values="total_score")
    .loc[list(CALIBRATORS), foldOrder]
  )
  variantDelta = variantPivot - variantPivot.loc[CONTROL]
  sensitivityDeltas[variant] = variantDelta
  for name in CALIBRATORS:
    if name == CONTROL:
      continue
    meanGap, positive, count, verdict = judge(variantDelta.loc[name], foldOrder)
    sensitivityRows.append({
      "OOF 범위": variant, "후보": name, "Δ 평균": meanGap,
      "부호 +": f"{positive}/{count}", "판정": verdict,
      "b (W_2022_2023)": (
        float(fittedCalibrations[("W_2022_2023", variant, name)].params.get("b", np.nan))
        if "b" in fittedCalibrations[("W_2022_2023", variant, name)].params else np.nan
      ),
    })

sensitivityFrame = pd.DataFrame(sensitivityRows)
verdictPivot = sensitivityFrame.pivot(index="후보", columns="OOF 범위", values="판정")[OOF_VARIANTS]
gapPivot = sensitivityFrame.pivot(index="후보", columns="OOF 범위", values="Δ 평균")[OOF_VARIANTS]
gapPivot["차이"] = gapPivot["마지막 블록"] - gapPivot["3블록"]

print("판정 대조 — 두 OOF 범위")
display(verdictPivot)
print("fold 평균 Δ 대조")
display(gapPivot)

verdictsAgree = bool((verdictPivot["3블록"] == verdictPivot["마지막 블록"]).all())
bestAgrees = bool(verdictPivot.loc[bestCandidate, "3블록"] == verdictPivot.loc[bestCandidate, "마지막 블록"])
print(f"\n모든 후보의 판정이 같은가        : {verdictsAgree}")
print(f"Δ 평균 최고 후보({bestCandidate})의 판정이 같은가 : {bestAgrees}")
print(f"Δ 평균 차이의 최대 절대값       : {float(gapPivot['차이'].abs().max()):.6f}"
      f"  (문턱의 {float(gapPivot['차이'].abs().max()) / RANK_THRESHOLD:.2f}배)")
print()
print("=== 민감도 판정 ===")
if verdictsAgree:
  print("  OOF 분할 범위를 바꿔도 판정이 같다 → 이 축이 결론을 만들지 않았다")
else:
  differing = verdictPivot.index[verdictPivot["3블록"] != verdictPivot["마지막 블록"]].tolist()
  print(f"  판정이 갈린 후보 : {differing}")
  print(f"  이 후보들은 채택하지 않는다 — 결론이 OOF 분할 선택에 의존한다는 뜻이다")

fig, axis = plt.subplots(figsize=(11.5, 5.0))
variantOffset = 0.2
for offset, variant, color in [(-variantOffset, "3블록", "#4c72b0"), (variantOffset, "마지막 블록", "#dd8452")]:
  axis.bar(np.arange(len(candidateNames)) + offset,
           [float(gapPivot.loc[name, variant]) for name in candidateNames],
           2 * variantOffset, label=f"OOF {variant}", color=color, zorder=3)
axis.axhline(0, color="#333333", linewidth=1, zorder=4)
axis.axhline(RANK_THRESHOLD, color="#c44e52", linestyle="--", linewidth=1.4, zorder=4,
             label=f"순위 문턱 {RANK_THRESHOLD}")
axis.set_xticks(np.arange(len(candidateNames)))
axis.set_xticklabels(candidateNames, rotation=15, ha="right")
axis.set_ylabel("fold 평균 Δ (통제군 대비)")
axis.set_title("그림 6. OOF 분할 범위를 바꿔도 판정이 유지되는가", fontsize=12.5)
sensitivitySpan = gapPivot[OOF_VARIANTS].to_numpy()
axis.set_ylim(min(float(sensitivitySpan.min()) * 1.35, -RANK_THRESHOLD * 0.4),
              max(float(sensitivitySpan.max()) * 1.28, RANK_THRESHOLD * 1.8))
axis.legend(fontsize=9.5, loc="upper left")
plt.tight_layout()
plt.show()

---

## 9. 개선분은 정말 계단에서 왔는가, 그리고 천장까지의 간격은 무엇인가

두 가지를 확인한다.

**밴드 이동.** 보정이 단가 밴드를 몇 행이나 옮겼고 순증이 얼마인가. 노트북 12가 S1→S2
제출에 했던 분석이며 그때는 21.0%가 이동하고 순증이 이동의 11.6%였다. 보정에서도
이동 대비 순증이 작다면 신호와 잡음을 함께 밀고 있는 것이다.

**천장까지의 간격을 세 층으로 쪼갠다.**

| 층 | 무엇인가 | 누수 |
|----|----------|------|
| OOF fit Δ | 실제 후보 — 학습 창 내부 OOF로 fit | 없음 |
| 누수 fit Δ | 같은 함수 형태를 **검증 fold에 직접** fit | **있음 — 진단 전용** |
| 오라클 상한 Δ | 행마다 이상적인 밀기 | 원리상 불가능 |

`OOF fit`과 `누수 fit`의 간격은 **표본 밖 fit의 어려움**이다. `누수 fit`과 `오라클 상한`의
간격은 **함수 형태의 한계**다. 앞이 크면 fit 방법을, 뒤가 크면 함수 형태를 의심해야 한다.
둘 다 작고 상한도 작으면 이 방향에 남은 것이 없다.

In [ ]:
bandRows = []
for foldName in foldOrder:
  calibration = fittedCalibrations[(FOLDS[foldName].train_window, "3블록", bestCandidate)]
  frame = band_transition(baseTermsByFold[foldName], calibration)
  row = frame.loc["합계"].to_dict()
  row["fold"] = foldName
  row["Δ total"] = float(deltaVsControl.loc[bestCandidate, foldName])
  bandRows.append(row)
bandFrame = pd.DataFrame(bandRows).set_index("fold")[
  ["평가 행", "밴드 이동", "좋아짐", "나빠짐", "순증", "이동 비율", "Δ total"]
]
print(f"단가 밴드 이동 — {bestCandidate} 적용 전후")
display(bandFrame)
movedTotal = int(bandFrame["밴드 이동"].sum())
netTotal = int(bandFrame["순증"].sum())
print(f"\n총 이동 {movedTotal:,}행 · 순증 {netTotal:,}행"
      f" (이동의 {netTotal / movedTotal:.1%})" if movedTotal else "\n밴드 이동이 0행이다")
print(f"노트북 12의 S1→S2 제출 비교 : 이동 21.0% · 순증은 이동의 11.6%")

leakyRows = []
leakyStarted = time.perf_counter()
for foldName in foldOrder:
  terms = baseTermsByFold[foldName]
  base = score_normalized(terms)
  for name in candidateNames:
    # 검증 구간에 직접 fit — 설계서 06 2.2절 위반이며 상한 진단으로만 쓴다.
    leakyCalibration = fit_calibration(name, terms)
    leakyRows.append({
      "fold": foldName, "후보": name,
      "OOF fit Δ": float(deltaVsControl.loc[name, foldName]),
      "누수 fit Δ": score_normalized(terms, leakyCalibration) - base,
    })
leakyFrame = pd.DataFrame(leakyRows)
print(f"\n누수 fit {len(leakyFrame)}회 · {time.perf_counter() - leakyStarted:.1f}s")

bestShiftBudget = 0.0
for foldName in foldOrder:
  calibration = fittedCalibrations[(FOLDS[foldName].train_window, "3블록", bestCandidate)]
  for predicted, _ in baseTermsByFold[foldName].values():
    if predicted.size:
      bestShiftBudget = max(bestShiftBudget, float(np.abs(calibration(predicted) - predicted).max()))

layerRows = []
for name in candidateNames:
  subset = leakyFrame[leakyFrame["후보"] == name]
  layerRows.append({
    "후보": name,
    "OOF fit Δ 평균": float(subset["OOF fit Δ"].mean()),
    "누수 fit Δ 평균": float(subset["누수 fit Δ"].mean()),
    "표본 밖 손실": float(subset["누수 fit Δ"].mean() - subset["OOF fit Δ"].mean()),
  })
layerFrame = pd.DataFrame(layerRows).set_index("후보")
oracleAtBudget = float(np.mean([
  oracle_row_bound(baseTermsByFold[foldName], bestShiftBudget) - score_normalized(baseTermsByFold[foldName])
  for foldName in foldOrder
]))
layerFrame["오라클 상한 Δ 평균"] = oracleAtBudget
layerFrame["함수 형태 손실"] = layerFrame["오라클 상한 Δ 평균"] - layerFrame["누수 fit Δ 평균"]
print(f"\n세 층 비교 — 오라클은 {bestCandidate}의 실제 최대 밀기 폭 δ={bestShiftBudget:.4f} 기준")
display(layerFrame)
print(f"순위 문턱 {RANK_THRESHOLD} 대비")
print(f"  OOF fit 최고    : {layerFrame['OOF fit Δ 평균'].max():.6f}"
      f" (문턱의 {layerFrame['OOF fit Δ 평균'].max() / RANK_THRESHOLD:.2f}배)")
print(f"  누수 fit 최고   : {layerFrame['누수 fit Δ 평균'].max():.6f}"
      f" (문턱의 {layerFrame['누수 fit Δ 평균'].max() / RANK_THRESHOLD:.2f}배)")
print(f"  오라클 상한     : {oracleAtBudget:.6f}"
      f" (문턱의 {oracleAtBudget / RANK_THRESHOLD:.2f}배)")

왼쪽이 **표본 밖 fit의 손실**이다. `OOF fit`과 `누수 fit`을 나란히 두었고, 오라클 상한은
한 자릿수 이상 클 수 있어 같은 막대로 그리면 앞의 둘이 눌리므로 선이나 주석으로 낸다
(수치는 위 표에 있다). 오른쪽이 밴드 이동이며, 초록과 빨강의 차이가 순증이다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2), gridspec_kw={"width_ratios": [1.1, 1]})

layerPositions = np.arange(len(candidateNames))
layerWidth = 0.36
# 오라클 상한은 후보 Δ보다 한 자릿수 이상 클 수 있어 같은 막대로 그리면 앞의 둘이 눌린다.
# 비교의 요점은 OOF fit과 누수 fit의 간격이므로 그 둘만 막대로 두고 상한은 선·주석으로 낸다.
for offset, column, color in [
  (-layerWidth / 2, "OOF fit Δ 평균", "#4c72b0"),
  (layerWidth / 2, "누수 fit Δ 평균", "#dd8452"),
]:
  axes[0].bar(layerPositions + offset, [float(layerFrame.loc[name, column]) for name in candidateNames],
              layerWidth, label=column, color=color, zorder=3)
axes[0].axhline(RANK_THRESHOLD, color="#111111", linestyle="--", linewidth=1.4, zorder=4,
                label=f"순위 문턱 {RANK_THRESHOLD}")
axes[0].axhline(0, color="#333333", linewidth=1, zorder=4)

fitLayers = layerFrame[["OOF fit Δ 평균", "누수 fit Δ 평균"]].to_numpy()
layerTop = max(float(fitLayers.max()) * 1.55, RANK_THRESHOLD * 2.2)
if oracleAtBudget <= layerTop:
  axes[0].axhline(oracleAtBudget, color="#c44e52", linestyle=":", linewidth=1.8, zorder=4,
                  label=f"오라클 상한 {oracleAtBudget:.5f}")
else:
  axes[0].annotate(
    f"오라클 상한 {oracleAtBudget:.5f} (문턱의 {oracleAtBudget / RANK_THRESHOLD:.1f}배) — 축 밖",
    xy=(0.02, 0.94), xycoords="axes fraction", fontsize=9, color="#c44e52",
  )
axes[0].set_ylim(min(float(fitLayers.min()) * 1.3, -RANK_THRESHOLD * 0.4), layerTop)
axes[0].set_xticks(layerPositions)
axes[0].set_xticklabels(candidateNames, rotation=15, ha="right")
axes[0].set_ylabel("fold 평균 Δ")
axes[0].set_title("표본 밖 fit의 손실 — OOF fit 과 누수 fit 의 간격", fontsize=11.5)
axes[0].legend(fontsize=8.5, loc="upper right")

axes[1].bar(positions - 0.2, bandFrame["좋아짐"], 0.4, label="좋아짐", color="#55a868", zorder=3)
axes[1].bar(positions + 0.2, -bandFrame["나빠짐"], 0.4, label="나빠짐", color="#c44e52", zorder=3)
axes[1].plot(positions, bandFrame["순증"], marker="o", color="#111111", linewidth=1.8,
             label="순증", zorder=5)
axes[1].axhline(0, color="#333333", linewidth=1, zorder=4)
axes[1].set_xticks(positions)
axes[1].set_xticklabels(foldTickLabels)
axes[1].set_ylabel("행 수")
axes[1].set_xlabel("fold · 검증 연도")
axes[1].set_title(f"{bestCandidate} 의 단가 밴드 이동", fontsize=11.5)
axes[1].legend(fontsize=9)

fig.suptitle("그림 7. 천장까지의 간격과 밴드 이동", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

---

## 10. 결론

아래 표는 **코드가 만든다.** 노트북 15에서 markdown 주장이 출력과 어긋난 사례가
있었으므로(7-4절에 기록), 판정 문장을 사람이 쓰지 않고 계산 결과에서 뽑는다.

In [ ]:
yearsCovered = sorted({spec.valid_year for spec in FOLDS.values()})
oracleCrossesThreshold = bool(
  float(oracleFrame[[column for column in oracleFrame.columns if column.startswith("δ=")]].mean().max())
  > RANK_THRESHOLD
)

conclusion = pd.DataFrame({
  "항목": [
    "F0 재현 게이트 (1e-09)",
    "terms 경로 = 프레임 경로 (1e-09)",
    "M6 전제 — FICR이 개선분을 주도하는 fold",
    "FICR 기여율 중앙값",
    "경계 ±1%p 밀도 범위",
    "오라클 상한이 문턱을 넘는 δ가 있는가",
    "오라클 상한 (실제 밀기 폭 기준)",
    "우위 확정 후보",
    "잠정 우위 후보",
    "Δ 평균 최고 후보",
    "그 후보의 Δ 평균 · 부호",
    "그 후보의 bootstrap P 범위",
    "2024 fold만 봤다면 우위 확정이었을 후보",
    "OOF 분할 민감도 — 판정 일치",
    "목적함수가 갈린 학습 창",
    "검증 연도",
  ],
  "값": [
    str(reproductionOk),
    str(pathOk),
    f"{ficrLedFolds}/{len(foldOrder)}",
    f"{decomposeFrame['FICR 기여율'].median():.1%}",
    f"{densityFrame['경계 ±1%p 비율'].min():.1%} ~ {densityFrame['경계 ±1%p 비율'].max():.1%}",
    str(oracleCrossesThreshold),
    f"{oracleAtBudget:.6f} (문턱의 {oracleAtBudget / RANK_THRESHOLD:.2f}배)",
    ", ".join(robustCandidates) or "없음",
    ", ".join(tentativeCandidates) or "없음",
    bestCandidate,
    f"{float(deltaVsControl.loc[bestCandidate].mean()):+.6f}"
    f" (문턱의 {abs(float(deltaVsControl.loc[bestCandidate].mean())) / RANK_THRESHOLD:.2f}배)"
    f" · {int((deltaVsControl.loc[bestCandidate] > 0).sum())}/{len(foldOrder)}",
    f"{bootstrapFrame['P(보정 > 무보정)'].min():.3f} ~ {bootstrapFrame['P(보정 > 무보정)'].max():.3f}",
    ", ".join([n for n in judgeFrame.index if judgeFrame.loc[n, "판정 (2024만)"] == "우위 확정"]) or "없음",
    str(verdictsAgree),
    f"{len(windowOrder) - identicalWindows}/{len(windowOrder)}",
    f"{len(yearsCovered)}개 {yearsCovered}",
  ],
}).set_index("항목")
display(conclusion)

# Decision Box ㉙의 채택 조건을 코드로 판정한다. 조건은 결과를 보기 전에 정해 뒀다.
adoptionChecks = {
  "① 부호가 아홉 fold 전부 양수": int((deltaVsControl.loc[bestCandidate] > 0).sum()) == len(foldOrder),
  "② Δ 평균이 순위 문턱 초과": float(deltaVsControl.loc[bestCandidate].mean()) > RANK_THRESHOLD,
  "③ OOF 분할 민감도에서 판정 일치": bestAgrees,
  "④ F0 재현 게이트 통과": reproductionOk,
  "⑤ terms·프레임 경로 일치": pathOk,
}
print(f"\n=== Decision Box ㉙ 채택 조건 판정 — 대상 {bestCandidate} ===")
for label, passed in adoptionChecks.items():
  print(f"  {'통과' if passed else '실패'}  {label}")
adopt = all(adoptionChecks.values())
print(f"\n  전체 판정 : {'(C) 채택' if adopt else '(A) 채택하지 않는다'}")
print(f"  근거      : {'다섯 조건을 모두 통과했다' if adopt else '위 실패 항목이 하나라도 있으면 채택하지 않는다'}")
print()
print("이 판정은 사전 등록된 규칙의 결과다. 규칙을 결과에 맞춰 고치지 않는다")

### Decision Box ㉙ — 보정을 채택할 조건을 결과 이전에 못 박는다

**이 결정표는 실행 전에 작성했다.** calibration은 후보를 만들기 쉽고 지표가 계단 함수라
사후에 규칙을 고르면 거의 항상 "무언가는 개선됐다"고 말할 수 있다. 그래서 조건을 먼저
적고 위 셀이 그것을 기계적으로 판정하게 한다.

**선택지**

- (A) 보정을 채택하지 않고 `lgbm_pooled` 원본을 유지한다
- (B) 보정을 채택하되 제출 후보로만 남긴다
- (C) 보정을 작업 모델에 포함시킨다

**채택 조건 — 다섯 개 전부를 통과해야 (C)다**

| 조건 | 근거 |
|------|------|
| ① Δ 부호가 아홉 fold 전부 양수 | 설계서 06 5.4절 셋째 칸 — 부호가 흔들리면 크기는 보지 않는다 |
| ② Δ 평균 > 0.0036 | 첫째 칸 — 측정된 fold 간 변동보다 커야 한다 |
| ③ OOF 분할 민감도에서 판정 일치 | 결론이 우리가 고른 분할의 성질이면 안 된다 (8절) |
| ④ F0 재현 게이트 통과 | 실패하면 노트북 14·15와 숫자를 나란히 놓을 수 없다 |
| ⑤ terms·프레임 경로 일치 | 갈리면 이 노트북의 숫자와 제출물이 다른 것을 가리킨다 |

①②를 통과하고 ③이 어긋나면 (A)다 — 크기와 부호가 맞아도 그것이 분할 선택의 산물이면
채택 근거가 아니다. ①만 통과하고 ②가 미달이면 설계서 06 5.4절 둘째 칸의 **잠정 채택**에
해당하지만, 이 노트북은 그 경우도 **(A)로 처리한다.** 보정은 파이프라인에 상시 붙는
변환이므로 잠정 근거로 넣으면 이후 모든 실험이 그 위에서 이뤄진다. 되돌리기 비용이
피처 하나를 넣고 빼는 것과 다르다.

**제출은 이번 작업에서 하지 않는다.** 남은 제출 3회를 후보마다 한 번씩 쓰면 설계서 06
5.4절이 경계하는 리더보드 자동 선택에 그대로 걸린다. 제출 후보 결정은 M6 결과를 포함해
따로 판정한다.

**채택 결과와 그 해석은 실행 후 이 자리에 기록한다.** 위 셀의 `Decision Box ㉙ 채택 조건
판정` 출력이 근거가 되며, 조건은 위 표에서 바꾸지 않는다.

---

## 산출물 확인

In [ ]:
print("이 노트북이 쓴 파일 : 없음 (분석 전용)")
print(f"학습한 모델        : {len(timingFrame)}회 (메모리에만 존재, 저장하지 않음)")
print(f"fit한 보정         : {len(fitFrame)}개 (창 {len(windowOrder)} × OOF 범위 {len(OOF_VARIANTS)} × 후보 {len(CALIBRATORS)})")
print(f"총 학습 시간       : {timingFrame['초'].sum():.0f}s")
print()
print(f"F0 재현 게이트 통과              : {reproductionOk}")
print(f"terms·프레임 경로 일치           : {pathOk}")
print(f"OOF 분할 겹침 행                 : {int(oofFrame['겹침 행'].sum())}")
print(f"우위 확정 후보                   : {robustCandidates or '없음'}")
print(f"잠정 우위 후보                   : {tentativeCandidates or '없음'}")
print(f"Decision Box ㉙ 판정             : {'(C) 채택' if adopt else '(A) 채택하지 않는다'}")

이 노트북은 파일을 만들지 않는다. 모델도 제출물도 저장하지 않으며 원자료와 라벨은
읽기만 했다.

**실행 후 채울 것** — 아래 세 곳은 결과를 받은 뒤 이 노트북과 설계서에 함께 기록한다.

1. 0절의 `PINNED_COMMIT` — 이 실행이 실제로 쓴 커밋 SHA
2. 10절 Decision Box ㉙ — 채택 결과와 그 해석
3. 설계서 04 7.1절 M6 상태, 설계서 06 5.4절 판정표의 새 행

**다음 단계는 결과에 달려 있다.** 보정이 채택되면 남은 후보는 설계서 04 7.1절의
M4(P1 물리·공간 피처)와 M5(fold OOF 앙상블)다. 채택되지 않으면 이 노트북의 5절
오라클 상한이 그 판단의 근거가 된다 — 상한 자체가 낮았다면 정산 구간을 겨냥하는
방향에 남은 것이 없다는 뜻이고, 상한은 높은데 후보가 못 가져왔다면 함수 형태를
넓히는 것이 다음 후보가 된다. 어느 쪽인지는 9절의 세 층 비교가 가른다.